In [20]:
import os
from typing import List, Dict
import csv
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.core import Feedback
from trulens.providers.openai import OpenAI as tru_openai
from dotenv import load_dotenv
from trulens.apps.custom import instrument
from openai import OpenAI


In [21]:
# with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
#     csv_reader = csv.DictReader(file)
#     # Iterate through rows as dictionaries
#     queries = []
#     for row in csv_reader:
#         queries.append(row["query"]) 

with open("../../GroundTruths_Dataset -No Multihop No yes or no questions.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 

In [22]:
len(queries)

15

In [23]:
load_dotenv()


True

In [24]:
from trulens.core import TruSession


session = TruSession()

## Uncomment the following to reset database 
# session.reset_database()

In [25]:
# pip install pinecone[grpc]
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [26]:
llm = OpenAI(api_key=os.getenv("OPEN_AI_EVAL_KEY"))

In [27]:
embed = llm.embeddings.create


In [28]:
prompt ="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points"


In [29]:
class retriever:
        def __init__(self, embed, index, llm):
             self.embed = embed
             self.index = index
             self.llm = llm
        def get_data(self,query):
            hypothetical_answer = self._augment_answer(query)
            embedding=self.embed( model="text-embedding-3-large",input=(query+hypothetical_answer)).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=5,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            return docs
        
        def _augment_answer(self, query):
             return self.llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a useful assistant for an organization. Youre mission is to provide a short and useful answer regarding a provided query"},
        {
            "role": "user",
            "content": query
        }
    ]
).choices[0].message.content


In [30]:
class generator:
    def __init__(self, llm):
        self.llm = llm
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": query+formatted_context
        }
    ]
)
        return response.choices[0].message

In [31]:
ret = retriever(embed, index, llm)
gen = generator(llm)


In [13]:
# ret.get_data("What are the requirement documents for the good standing certificate of medical staff in the sector the is fee-exempt for renewal staff licenses?")

['if the applicant works in one of the private outsourcing companies contracting with the Emirates Health Services Corporation, he must apply for a service card to obtain a certificate of good standing and professional conduct in the government sectorRequired DocumentsMedical staff in the private sector:A copy of medical profession practice card (electronic license issued by MOHAP)A copy of a valid passportA copy of a valid passportAn experience certificate issued from the private medical facility, approved by the medical',
 'passportA copy of a valid passportAn experience certificate issued from the private medical facility, approved by the medical directorIn case of license cancellation please attach the cancellation letter from MOHAPMedical staff in the government sector:A letter of experience from the Department of Human Resources - Ministry of Health and Prevention or Emirates Health Services from BayanatiAn electronic experience certificate for licensing purposes, signed by the m

In [32]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response.content

In [33]:
rag_app = Rag_app(gen, ret)


In [16]:
# rag_app.query("What are the requirement documents for the good standing certificate of medical staff in the sector the is fee-exempt for renewal staff licenses? ")

"The context provided outlines the specific requirements for obtaining a good standing certificate for medical staff, distinguishing between those in the government and private sectors. For medical staff in the private sector, a valid passport copy, an electronic license issued by MOHAP, and an experience certificate from the medical facility, approved by the medical director, are needed. In case of cancellation of a license, the cancellation letter from MOHAP must be attached. Additionally, a letter from the medical facility requesting the re-licensing of the doctor, a copy of the doctor's valid license, an assessment certificate, the facility's plan, and, if applicable, a medical fitness certificate for those 60 years or older are required. For government sector staff, documentation includes a letter of experience from the Department of Human Resources or Emirates Health Services and an electronic experience certificate for licensing purposes, signed by the medical or technical direc

In [34]:
rag_app = Rag_app(gen, ret)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [37]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="Query expansion RAG",
    app_version="4o-large_3-500-simple_questions",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

Function <function Rag_app.generate at 0x0000013E11168E00> was not found during instrumentation walk. Make sure it is accessible by traversing app <__main__.Rag_app object at 0x0000013E109CE660> or provide a bound method for it as TruCustomApp constructor argument `methods_to_instrument`.
Function <function Rag_app.retrieve at 0x0000013E1116AE80> was not found during instrumentation walk. Make sure it is accessible by traversing app <__main__.Rag_app object at 0x0000013E109CE660> or provide a bound method for it as TruCustomApp constructor argument `methods_to_instrument`.
Function <function Rag_app.query at 0x0000013E1116A8E0> was not found during instrumentation walk. Make sure it is accessible by traversing app <__main__.Rag_app object at 0x0000013E109CE660> or provide a bound method for it as TruCustomApp constructor argument `methods_to_instrument`.


In [19]:
# rag_app.query(queries[0])

In [38]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [39]:
with tru_rag as recording:
    for eval in queries:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


How do I register for controlled or semi-controlled drugs custody?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


What are the requirements for renewing the registration of a conventional pharmaceutical product?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


How do I appeal a decision made by the Medical Licensing Committee?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


What is the process for obtaining a certificate of amendment for registered pharmaceutical products?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Error calling wrapped function generate.
Traceback (most recent call last):
  File "c:\Users\abdal\Desktop\RAG-main\env\Lib\site-packages\trulens\core\instruments.py", line 769, in tru_wrapper
    rets, tally = core_endpoint.Endpoint.track_all_costs_tally(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File 

KeyboardInterrupt: 

In [19]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://192.168.1.12:25854 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>